# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [6]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [7]:
# I am choosing "Managing Oneself" by Peter Drucker - a classic article on career management
# Using LangChain's PyPDFLoader to load the PDF

from langchain_community.document_loaders import PyPDFLoader
import requests
import os
import tempfile

# Download and load the PDF
pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

# Download PDF to temp file
response = requests.get(pdf_url)
with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
    tmp_file.write(response.content)
    tmp_path = tmp_file.name

# Load using PyPDFLoader
loader = PyPDFLoader(tmp_path)
docs = loader.load()

# Join all pages into single document text
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Clean up temp file
os.unlink(tmp_path)

print(f"Loaded {len(docs)} pages")
print(f"Total characters: {len(document_text)}")
print(f"\nFirst 500 characters:\n{document_text[:500]}")

Loaded 13 pages
Total characters: 51452

First 500 characters:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [10]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# Initialize OpenAI client with course API gateway
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# Define Pydantic model for structured output
class DocumentSummary(BaseModel):
    Author: str = Field(description="The author of the document")
    Title: str = Field(description="The title of the document")
    Relevance: str = Field(description="A statement explaining why this article is relevant for an AI professional")
    Summary: str = Field(description="A concise summary no longer than 1000 tokens")
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens")
    OutputTokens: int = Field(description="Number of output tokens")

INSTRUCTIONS = """You are a scholarly assistant that summarizes academic and professional articles.
Your summaries should be written in Bureaucratese - the obscure, formal, and convoluted language of bureaucrats.
Use passive voice, complex sentence structures, jargon, and formal hedging language.
Example phrases: "pursuant to", "in accordance with", "it has been determined that", "with respect to the aforementioned"."""

USER_PROMPT = """Please analyze the following document and provide a structured summary.

<document>
{document}
</document>

Requirements:
1. Identify the author and title
2. Explain in one paragraph why this article is relevant for AI professionals in their career development
3. Write a concise summary (max 1000 tokens) using Bureaucratese style - formal, convoluted bureaucratic language
4. Note the tone used"""

# Make API request with structured output
response = client.responses.parse(
    model="gpt-4o-mini",  
    input=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT.format(document=document_text)}
    ],
    text_format=DocumentSummary,
    temperature=0.7 # for the space of creativity to the model as we have learned.
)

# Parse the response
parsed_output = response.output_parsed

# Add token counts from response
summary_result = DocumentSummary(
    Author=parsed_output.Author,
    Title=parsed_output.Title,
    Relevance=parsed_output.Relevance,
    Summary=parsed_output.Summary,
    Tone=parsed_output.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)

# Print the structured Pydantic BaseModel object
print(summary_result)

# Display results in a readable format
from IPython.display import display, Markdown

display(Markdown(f"## {summary_result.Title}"))
display(Markdown(f"**Author:** {summary_result.Author}"))
display(Markdown(f"**Tone:** {summary_result.Tone}"))
display(Markdown(f"\n**Relevance for AI Professionals:**\n{summary_result.Relevance}"))
display(Markdown(f"\n**Summary (Bureaucratese):**\n{summary_result.Summary}"))
display(Markdown(f"\n**Token Usage:** Input: {summary_result.InputTokens}, Output: {summary_result.OutputTokens}"))

Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is particularly pertinent for AI professionals engaged in the rapidly evolving landscape of knowledge work, as it elucidates the necessity for self-management and self-awareness in navigating career trajectories that increasingly require individuals to take proactive responsibility for their professional development, thereby enhancing their effectiveness and adaptability in a competitive field.' Summary="In the contemporary milieu characterized by unprecedented opportunities for those possessing ambition, intellect, and determination, it has been posited that individuals must assume the role of Chief Executive Officers of their own careers, particularly within the domain of knowledge work. The exigency of managing oneself necessitates the cultivation of an acute self-awareness encompassing one's strengths, weaknesses, preferred work styles, values, and optimal environments for contribution. It has been determine

## Managing Oneself

**Author:** Peter F. Drucker

**Tone:** Bureaucratic, formal, convoluted.


**Relevance for AI Professionals:**
This article is particularly pertinent for AI professionals engaged in the rapidly evolving landscape of knowledge work, as it elucidates the necessity for self-management and self-awareness in navigating career trajectories that increasingly require individuals to take proactive responsibility for their professional development, thereby enhancing their effectiveness and adaptability in a competitive field.


**Summary (Bureaucratese):**
In the contemporary milieu characterized by unprecedented opportunities for those possessing ambition, intellect, and determination, it has been posited that individuals must assume the role of Chief Executive Officers of their own careers, particularly within the domain of knowledge work. The exigency of managing oneself necessitates the cultivation of an acute self-awareness encompassing one's strengths, weaknesses, preferred work styles, values, and optimal environments for contribution. It has been determined that one may achieve genuine excellence solely by leveraging one's intrinsic strengths in conjunction with a profound understanding of one's operational modalities. To facilitate this process, a systematic approach known as feedback analysis is recommended, wherein individuals document anticipated outcomes of significant decisions and subsequently compare these expectations with actual results to discern patterns indicative of personal competencies. Furthermore, it is imperative that individuals ascertain their preferred methods of learning and working, as well as the ethical frameworks they espouse. A thorough recognition of one's values is essential, for alignment between personal and organizational values is paramount to mitigating frustration and enhancing performance. In determining one's place within the professional ecosystem, it is suggested that individuals must decisively reject roles misaligned with their capacities and values while actively seeking environments conducive to their strengths. Moreover, the article underscores the importance of contributions to organizational objectives, delineating a shift from the historical paradigm wherein contributions were externally dictated to one where knowledge workers must proactively define their value-add. Finally, it is imperative to acknowledge the responsibility for fostering effective interpersonal relationships, which necessitates an understanding of colleagues' strengths and communication styles. This comprehensive self-management paradigm is particularly salient for those navigating the second half of their professional lives, where the potential for contributing meaningfully persists beyond traditional career structures, thereby heralding a transformative approach to personal and professional fulfillment.


**Token Usage:** Input: 12481, Output: 445

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Evaluation Approach

Using DeepEval library with:
- **SummarizationMetric**: 5 custom assessment questions targeting key concepts from "Managing Oneself"
- **G-Eval Coherence**: 5 evaluation steps for logical structure
- **G-Eval Tonality**: 5 steps to verify Bureaucratese style consistency
- **G-Eval Safety**: 5 steps for professional appropriateness

In [ ]:
# will do after the coverage of the topic of evaluation and testing.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
